In [ ]:
import pandas as pd
import numpy as np

# Assuming df_final is loaded from the previous phase
df_final = pd.read_csv("./data/intermi/final_chess_dataset.csv")
print("=" * 50)
print("BEFORE CLEANING SUMMARY")
print("=" * 50)
print(f"Initial Shape: {df_final.shape[0]:,} rows × {df_final.shape[1]} columns")

# Focus on the specific columns we know have issues
cols_to_check = ['white_acl', 'white_castled', 'num_moves', 'max_white_advantage']
print("\nMissing values before:")
print(df_final[cols_to_check].isnull().sum())

print("\nOutlier Check (Minimums & Maximums):")
print(df_final[['num_moves', 'white_acl', 'max_white_advantage']].agg(['min', 'max']))

In [ ]:
# 1. Impute the 18 missing Stockfish rows using the Median
stockfish_numeric_cols = [
    'white_acl', 'black_acl', 'white_blunders', 'black_blunders', 
    'white_mistakes', 'black_mistakes', 'final_eval', 
    'max_white_advantage', 'max_black_advantage', 'game_sharpness', 'acl_gap'
]

for col in stockfish_numeric_cols:
    median_val = df_final[col].median()
    df_final[col] = df_final[col].fillna(median_val)

# 2. Handle Kaggle-specific board features missing in Lichess data
# Categorical features -> 'Unknown'
categorical_board_cols = ['white_castled', 'black_castled', 'white_castle_side', 'black_castle_side']
for col in categorical_board_cols:
    df_final[col] = df_final[col].fillna('Unknown')

# Numeric features -> -1 (Distinct flag for missing)
df_final['num_captures'] = df_final['num_captures'].fillna(-1)

# Raw move strings -> 'Not Available'
df_final['moves_uci'] = df_final['moves_uci'].fillna('Not Available')
# Note: 'result' is missing for Lichess rows because it was renamed to 'winner_multiclass' and 'winner_binary'.
df_final['result'] = df_final['result'].fillna('Unknown')

print("Missing values handled successfully.")

In [ ]:
# 1. Remove Aborted / Ultra-short games (less than 2 full moves)
aborted_games_mask = df_final['num_moves'] < 2
aborted_count = aborted_games_mask.sum()
df_final = df_final[~aborted_games_mask].copy()

# 2. Cap extreme Stockfish evaluation scores (Winsorization)
# We clip extreme centipawn evaluations to a ceiling of 2000 and floor of -2000
cap_value = 2000

# Cap Final Eval and Max Advantages
eval_cols_to_cap = ['final_eval', 'max_white_advantage', 'max_black_advantage']
for col in eval_cols_to_cap:
    df_final[col] = df_final[col].clip(lower=-cap_value, upper=cap_value)

# Cap Average Centipawn Loss (ACL) - ACL is always positive
acl_cols_to_cap = ['white_acl', 'black_acl']
for col in acl_cols_to_cap:
    df_final[col] = df_final[col].clip(lower=0, upper=cap_value)

print(f"Removed {aborted_count} aborted games (moves < 2).")
print(f"Capped engine evaluations at ±{cap_value} centipawns.")

In [ ]:
print("=" * 50)
print("AFTER CLEANING SUMMARY")
print("=" * 50)
print(f"Final Cleaned Shape: {df_final.shape[0]:,} rows × {df_final.shape[1]} columns")

print("\nMissing values after cleaning (excluding target variables):")
# winner_binary is allowed to have missing values (Draws), so we exclude it from the check
remaining_missing = df_final.drop(columns=['winner_binary']).isnull().sum()
print(remaining_missing[remaining_missing > 0])

print("\nOutlier Check Resolved (Minimums & Maximums):")
print(df_final[['num_moves', 'white_acl', 'max_white_advantage', 'max_black_advantage']].agg(['min', 'max']))